# Few-Shot Learning untuk Sentiment Analysis

**Deskripsi**: Tutorial ini mendemonstrasikan *few-shot learning* — teknik yang memungkinkan model bahasa belajar dari **sangat sedikit contoh** (misalnya 8, 16, atau 32 sampel per kelas). Kita akan menggunakan **SetFit** (Sentence Transformer Fine-tuning) yang dirancang khusus untuk few-shot classification.

**Tujuan Pembelajaran**:
- Memahami konsep *few-shot learning* dan kapan menggunakannya
- Mampu menggunakan SetFit untuk few-shot classification
- Membandingkan performa dengan jumlah data sedikit vs banyak
- Membandingkan hasil zero-shot, few-shot, dan full fine-tuning
- Memahami trade-off antara jumlah data dan performa

---## 1. Instalasi Dependensi

In [ ]:
!pip install setfit transformers datasets scikit-learn seaborn matplotlib pandas tqdm torch evaluate

---## 2. Konsep Few-Shot Learning

**Apa itu Few-Shot Learning?**

Few-shot learning adalah teknik di mana model belajar dari jumlah contoh yang sangat terbatas (biasanya 1-100 sampel per kelas).

**SetFit** (*Sentence Transformer Fine-tuning*) adalah framework yang:
1. Menggunakan **Sentence Transformers** untuk menghasilkan embeddings kalimat
2. Melatih model dengan **contrastive learning** — membandingkan pasangan kalimat serupa vs berbeda
3. Menambahkan classifier head yang sederhana

**Keunggulan SetFit**:
- Bekerja sangat baik dengan 8-64 sampel per kelas
- Jauh lebih cepat dari fine-tuning model besar
- Hasilnya sering mendekati full fine-tuning

**Perbandingan dengan Zero-Shot**:
- Zero-shot: 0 sampel, performa lebih rendah, tidak perlu training
- Few-shot: 8-64 sampel, performa lebih tinggi, perlu training singkat
- Full fine-tuning: 1000+ sampel, performa tertinggi, perlu training lama

---## 3. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

from datasets import Dataset, DatasetDict, load_dataset
from setfit import SetFitModel, SetFitTrainer, Trainer

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

---## 4. Load Dataset Sentimen

In [ ]:
# Load dataset dari Hugging Face
dataset = load_dataset("sepidmnorozy/Indonesian_sentiment")

print(f"Train size: {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print(f"Test size: {len(dataset['test'])}")

---## 5. Membuat Few-Shot Dataset

Kita akan mengambil **N sampel per kelas** dari data training untuk mensimulasikan situasi few-shot.

In [ ]:
def create_few_shot_dataset(dataset, n_samples_per_class, split='train'):
    """Ambil n sampel per kelas untuk few-shot learning."""
    data = dataset[split]
    
    # Pisahkan berdasarkan label
    class_0 = [i for i, s in enumerate(data) if s['label'] == 0]
    class_1 = [i for i, s in enumerate(data) if s['label'] == 1]
    
    # Ambil n sampel per kelas
    np.random.seed(42)
    sampled_0 = np.random.choice(class_0, min(n_samples_per_class, len(class_0)), replace=False)
    sampled_1 = np.random.choice(class_1, min(n_samples_per_class, len(class_1)), replace=False)
    
    # Gabungkan
    sampled_indices = sorted(np.concatenate([sampled_0, sampled_1]))
    
    texts = [data[i]['text'] for i in sampled_indices]
    labels = [data[i]['label'] for i in sampled_indices]
    
    return Dataset.from_dict({'text': texts, 'label': labels})

# Buat dataset dengan berbagai jumlah sampel
label_map = {0: "NEGATIVE", 1: "POSITIVE"}

for n in [4, 8, 16, 32]:
    fewshot = create_few_shot_dataset(dataset, n, 'train')
    counts = fewshot.to_pandas()['label'].value_counts().sort_index()
    print(f"Few-shot ({n}/class): {len(fewshot)} total samples | "
          f"Neg: {counts.get(0, 0)}, Pos: {counts.get(1, 0)}")

---## 6. Model SetFit

SetFit menggunakan Sentence Transformer sebagai backbone. Kita akan menggunakan model `paraphrase-multilingual-MiniLM-L12-v2` yang mendukung Bahasa Indonesia.

In [ ]:
# Load model SetFit
model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    labels=["NEGATIVE", "POSITIVE"]
)

print(f"Model loaded: {model.model_name}")
print(f"Device: {model.device}")

---## 7. Eksperimen Few-Shot dengan 16 Sampel Per Kelas

Mari kita latih SetFit dengan hanya 16 sampel positif dan 16 sampel negatif.

In [ ]:
# Buat few-shot dataset: 16 sampel per kelas
N_SHOTS = 16
train_fs = create_few_shot_dataset(dataset, N_SHOTS, 'train')
eval_fs = dataset['validation']

print(f"Few-shot training set: {len(train_fs)} samples ({N_SHOTS} per class)")
print(f"Evaluation set: {len(eval_fs)} samples")

In [ ]:
# Inisialisasi trainer SetFit
trainer = SetFitTrainer(
    model=model,
    train_dataset=train_fs,
    eval_dataset=eval_fs,
    num_epochs=5,
    batch_size=16,
    learning_rate=2e-5,
    column_mapping={"text": "text", "label": "label"}
)

print(f"Trainer siap!")
print(f"  Batch size: {trainer.batch_size}")
print(f"  Epochs: {trainer.num_epochs}")
print(f"  Learning rate: {trainer.learning_rate}")

In [ ]:
# Latih model
print(f"Melatih SetFit dengan {len(train_fs)} sampel...")
start_time = time.time()

trainer.train()

elapsed = time.time() - start_time
print(f"Training selesai dalam {elapsed:.2f} detik! ✅")

In [ ]:
# Evaluasi pada test set
test_texts = [s['text'] for s in dataset['test']]
test_labels = [s['label'] for s in dataset['test']]

pred_labels = model(test_texts)

# Hitung metrik
accuracy = accuracy_score(test_labels, pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, pred_labels, average='macro')

print("=== Few-Shot (16/class) Results on Test Set ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f} (macro)")
print(f"Recall:    {recall:.4f} (macro)")
print(f"F1-Score:  {f1:.4f} (macro)")
print()
print(classification_report(test_labels, pred_labels, target_names=["NEGATIVE", "POSITIVE"]))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['NEGATIVE', 'POSITIVE'],
            yticklabels=['NEGATIVE', 'POSITIVE'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Few-Shot Sentiment ({N_SHOTS} samples/class)')
plt.show()

---## 8. Perbandingan: 4-shot, 8-shot, 16-shot, 32-shot

Mari kita lihat bagaimana performa berubah seiring bertambahnya jumlah sampel.

In [ ]:
shot_sizes = [4, 8, 16, 32]
results = []

test_texts = [s['text'] for s in dataset['test']]
test_labels = [s['label'] for s in dataset['test']]

for n in shot_sizes:
    print(f"\n{'='*50}")
    print(f"Few-Shot dengan {n} sampel per kelas...")
    
    # Buat model baru untuk setiap eksperimen
    exp_model = SetFitModel.from_pretrained(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        labels=["NEGATIVE", "POSITIVE"]
    )
    
    train_fs = create_few_shot_dataset(dataset, n, 'train')
    
    exp_trainer = SetFitTrainer(
        model=exp_model,
        train_dataset=train_fs,
        eval_dataset=dataset['validation'],
        num_epochs=5,
        batch_size=min(16, n*2),
        learning_rate=2e-5,
        column_mapping={"text": "text", "label": "label"}
    )
    
    start = time.time()
    exp_trainer.train()
    train_time = time.time() - start
    
    preds = exp_model(test_texts)
    acc = accuracy_score(test_labels, preds)
    _, _, f1_score, _ = precision_recall_fscore_support(test_labels, preds, average='macro')
    
    results.append({
        'shots': n,
        'accuracy': acc,
        'f1': f1_score,
        'train_time': train_time,
        'train_samples': len(train_fs)
    })
    
    print(f"  Accuracy: {acc:.4f} | F1: {f1_score:.4f} | Time: {train_time:.1f}s")

print("\n" + "="*50)
print("Semua eksperimen selesai! ✅")

In [ ]:
# Visualisasi perbandingan
df_results = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs Shot count
axes[0].plot(df_results['shots'], df_results['accuracy'], 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Shots (per class)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Few-Shot Accuracy vs Training Data Size')
axes[0].set_xticks(shot_sizes)
axes[0].grid(True, alpha=0.3)
for _, row in df_results.iterrows():
    axes[0].annotate(f'{row["accuracy"]:.3f}',
                    (row['shots'], row['accuracy']),
                    textcoords="offset points", xytext=(0, 10),
                    ha='center')

# F1 vs Shot count
axes[1].plot(df_results['shots'], df_results['f1'], 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Shots (per class)')
axes[1].set_ylabel('F1 Score (macro)')
axes[1].set_title('Few-Shot F1 Score vs Training Data Size')
axes[1].set_xticks(shot_sizes)
axes[1].grid(True, alpha=0.3)
for _, row in df_results.iterrows():
    axes[1].annotate(f'{row["f1"]:.3f}',
                    (row['shots'], row['f1']),
                    textcoords="offset points", xytext=(0, 10),
                    ha='center')

plt.suptitle('SetFit Few-Shot Performance on Indonesian Sentiment', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Tampilkan tabel perbandingan
print("=== Perbandingan Few-Shot Performance ===")
print(f"{'Shots (per class)':<20} {'Train Samples':<15} {'Accuracy':<12} {'F1 Score':<12} {'Train Time':<12}")
print("=" * 71)
for r in results:
    print(f"{r['shots']:<20} {r['train_samples']:<15} {r['accuracy']:.4f}       {r['f1']:.4f}       {r['train_time']:.1f}s")

---## 9. Uji Coba Inference

In [ ]:
# Uji coba dengan model 16-shot
test_samples = [
    "filmnya bagus banget, aktingnya mantap",
    "pelayanan sangat lambat dan makanannya tidak enak",
    "sayang banget barangnya rusak padahal baru dibeli",
    "recommended banget! puas dengan pelayanannya",
    "ga suka, kecewa dengan kualitasnya",
    "cukup lumayan untuk harganya"
]

print("=== Few-Shot Model Predictions ===")
print(f"{'Text':<55} {'Prediction':<12}")
print("=" * 67)
for text in test_samples:
    pred = model([text])[0]
    pred_label = "POSITIVE" if pred == 1 else "NEGATIVE"
    print(f"{text:<55} {pred_label:<12}")

---## 10. Perbandingan Akhir dengan Semua Pendekatan

In [ ]:
# Data perbandingan (dari hasil eksperimen di notebook lain)
# Catatan: isi dengan hasil aktual setelah menjalankan notebook

comparison = pd.DataFrame({
    'Approach': [
        'Zero-Shot (no training)',
        'Few-Shot (4/class)',
        'Few-Shot (8/class)',
        'Few-Shot (16/class)',
        'Few-Shot (32/class)',
        'Full Fine-Tuning (7,926 samples)'
    ],
    'Training Samples': [0, 8, 16, 32, 64, 7926],
    'Training Time': ['0s', '<10s', '<15s', '<30s', '<60s', '~5min'],
    'When to Use': [
        'No data, quick prototyping',
        'Minimal labeled data',
        'Small labeled dataset',
        'Moderate few-shot',
        'Upper few-shot range',
        'Maximum performance'
    ]
})

print("=== Perbandingan Semua Pendekatan ===")
print()
print(comparison.to_string(index=False))

---## 11. Kesimpulan

Dalam tutorial few-shot learning ini, kita telah mempelajari:

1. ✅ **Konsep few-shot learning** — belajar dari sedikit contoh
2. ✅ **SetFit framework** — efisien untuk few-shot classification
3. ✅ **Eksperimen multi-shot** — 4, 8, 16, 32 sampel per kelas
4. ✅ **Perbandingan performa** — semakin banyak data, semakin baik
5. ✅ **Evaluasi** — akurasi, F1, confusion matrix

**Insight Penting**:
- Bahkan dengan hanya **4-8 sampel per kelas**, SetFit bisa mencapai performa yang layak
- **16 sampel** biasanya merupakan sweet spot antara effort labeling dan performa
- **32+ sampel** mendekati performa full fine-tuning
- SetFit **jauh lebih cepat** (detik) dibanding fine-tuning model besar (menit)

**Rekomendasi**:
- Tidak punya data → Zero-Shot
- Punya 8-64 sampel → Few-Shot dengan SetFit
- Punya 1000+ sampel → Full Fine-Tuning

**Kapan Few-Shot Paling Berguna?**
- Dataset baru dengan sedikit label
- Domain spesifik dimana labeling mahal
- Eksplorasi cepat sebelum investasi labeling besar-besaran
- Situasi dimana akurasi 85-90% sudah cukup